# @zz-windfm-test/building-data-genome-project-2

Hourly meter readings from 1,636 non-residential buildings at 19 sites, with metadata and weather, 2016-2017.

Run all cells: **Runtime → Run all** (or ⌘/Ctrl + F9). The last code cell prints the endpoint address. This dataset ships files and no code, so that cell serves them with PowerAI's dataset runner, shown in full in the cell so you can read it first. It only reads: nothing in your files is run, and the rows it returns are capped.


In [ ]:
PRIMITIVE = "@zz-windfm-test/building-data-genome-project-2"
ARCHIVE = "https://hub-api.powerai.ai/api/artifacts/zz-windfm-test/building-data-genome-project-2/download-archive/"
WORKDIR = "/content/primitive"

import shutil, urllib.request, zipfile
from pathlib import Path

shutil.rmtree(WORKDIR, ignore_errors=True)          # re-runs start clean
Path(WORKDIR).mkdir(parents=True, exist_ok=True)

archive = Path("/content/primitive.zip")
urllib.request.urlretrieve(ARCHIVE, archive)
with zipfile.ZipFile(archive) as zf:
    zf.extractall(WORKDIR)

files = sorted(p for p in Path(WORKDIR).rglob("*") if p.is_file())
print(f"{PRIMITIVE}: {len(files)} files in {WORKDIR}")
for p in files[:20]:
    print(" ", p.relative_to(WORKDIR))
if len(files) > 20:
    print(f"  … and {len(files) - 20} more")


In [ ]:
import importlib.metadata as md, subprocess, sys
from pathlib import Path

print(f"Python {sys.version.split()[0]}")

# The file nearest the root first: a requirements.txt under docs/ or tests/ is not this
# primitive's, and rglob alone would pick whichever it met first.
found = sorted(
    Path(WORKDIR).rglob("requirements.txt"),
    key=lambda p: (len(p.relative_to(WORKDIR).parts), str(p)),
)
if not found:
    print("No requirements.txt. Install what you need with pip, e.g. !pip install pandas")
else:
    req = found[0]
    print(f"Installing {req.relative_to(WORKDIR)} …")
    if len(found) > 1:
        others = ", ".join(str(p.relative_to(WORKDIR)) for p in found[1:4])
        print(f"Not installed (only one file is): {others}")

    def versions():
        return {d.metadata["Name"].lower(): d.version for d in md.distributions() if d.metadata["Name"]}

    before = versions()
    done = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(req)],
        capture_output=True, text=True,
    )
    if done.returncode == 0:
        print("done")
        after = versions()
        # An install can replace what Colab ships; say so, so a later surprise has a cause.
        for name in sorted(n for n in before if n in after and before[n] != after[n])[:12]:
            print(f"  changed: {name} {before[name]} -> {after[name]}")
    else:
        print(
            "pip failed. pip installs the whole file or none of it, so one package that "
            "cannot be built on this Python stops all of it:"
        )
        print(done.stderr[-2000:])


In [ ]:
import subprocess, sys
from pathlib import Path

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn[standard]"],
    capture_output=True,
)
# The runner reads powerai_dataset.json (the list of tables) from the archive, and powerai_dataset_files.py says which files are tables.
Path(WORKDIR, "powerai_dataset_files.py").write_text('''"""Which files are tables, which files are partitions of one table, and what a table is called.

Standard library only, and no backslashes: the Hub's server imports it to decide what **Make
queryable** offers, and the launcher writes it into Colab beside the dataset runner, which uses it
for the files inside a zip. One copy, so the two cannot disagree about what a table is.
"""

from __future__ import annotations

import re
from pathlib import PurePosixPath
from typing import Any

#: What the runner reads, by file extension. A compressed delimited file is named ``x.csv.gz``.
FORMATS = {
    ".csv": "csv",
    ".tsv": "tsv",
    ".json": "json",
    ".jsonl": "jsonl",
    ".ndjson": "jsonl",
    ".xlsx": "xlsx",
    ".xls": "xls",
    ".parquet": "parquet",
    ".zip": "zip",
    ".sqlite": "sqlite",
    ".sqlite3": "sqlite",
    ".db": "sqlite",
    ".nc": "netcdf",
    ".nc4": "netcdf",
}
#: Formats that are a container: what is inside is found when the file is opened, in Colab, and
#: such a file is never grouped with others like it.
CONTAINERS = ("zip", "sqlite", "netcdf")
#: Formats whose name alone does not settle it: a ``.json`` is data only if it is a list of
#: records, and a ``.db`` only if it starts like a SQLite database.
SNIFFED_SUFFIXES = {".json": "json", ".db": "sqlite"}
COMPRESSION = (".gz", ".bz2", ".xz")
COMPRESSIBLE = ("csv", "tsv")

#: The most files that make one table. More are left out, and counted.
MAX_PARTS = 500

#: Files that are about the primitive, not its data: never offered as a table.
NOT_DATA_NAMES = frozenset(
    {
        "datapackage.json",
        "package.json",
        "package-lock.json",
        "agent.json",
        "composer.json",
        "tsconfig.json",
        "manifest.json",
        "metadata.json",
    }
)
NOT_DATA_PREFIXES = ("readme", "license", "licence", "powerai_", "changelog")


#: Documentation and media: not data, and never reported as a file the runner leaves alone.
PROSE_SUFFIXES = (
    ".txt",
    ".md",
    ".rst",
    ".pdf",
    ".png",
    ".jpg",
    ".jpeg",
    ".gif",
    ".svg",
    ".html",
    ".docx",
    ".ipynb",
    ".log",
)


def suffixes(path: str) -> tuple[str, str | None]:
    """The data suffix and the compression suffix (None when the file is not compressed)."""
    name = PurePosixPath(path).name.lower()
    for compression in COMPRESSION:
        if name.endswith(compression):
            return PurePosixPath(name[: -len(compression)]).suffix, compression
    return PurePosixPath(name).suffix, None


def is_about_the_primitive(path: str) -> bool:
    parts = PurePosixPath(path).parts
    name = parts[-1].lower()
    return (
        name in NOT_DATA_NAMES
        or name.startswith(NOT_DATA_PREFIXES)
        or name.endswith(".schema.json")
        or any(p.startswith(".") or p == "__MACOSX" for p in parts)
    )


def table_format(path: str) -> str | None:
    """The runner's name for this file's format, or None when it is not one it reads.

    Decided by the name alone; a ``.json`` is looked inside as well by whoever calls this,
    because most ``.json`` files in a primitive are configuration, not rows. ``zip`` is a
    container: what is in it is only known when it is opened, in Colab.
    """
    if is_about_the_primitive(path):
        return None
    suffix, compression = suffixes(path)
    fmt = FORMATS.get(suffix)
    if fmt is None:
        return None
    if compression and fmt not in COMPRESSIBLE:
        return None
    return fmt


def partition_key(path: str) -> str:
    """The path with each run of two or more digits made ``#``: years, months, part numbers.

    ``2022/prices.csv`` and ``2023/prices.csv`` share a key, and so do ``load_2021-01.csv`` and
    ``load_2021-02.csv``. A single digit is left alone, so ``table1.csv`` and ``table2.csv``,
    which are more likely two tables than two halves of one, do not.
    """
    return re.sub("[0-9]{2,}", "#", path.lower())


def group_files(files: list[tuple[str, int]]) -> tuple[list[dict[str, Any]], int]:
    """Files that are tables, grouped: ``[{"paths", "format", "size_bytes"}]`` and a count left out.

    Files that share a format and a ``partition_key`` are partitions of one table. A zip, a
    database and a gridded file are never grouped. At most ``MAX_PARTS`` files make one table; the rest are counted, not served.
    """
    keyed: dict[tuple[str, str], list[tuple[str, int]]] = {}
    singles: list[tuple[str, int, str]] = []
    for path, size in sorted(files):
        fmt = table_format(path)
        if fmt is None or not size:
            continue
        if fmt in CONTAINERS or "#" not in partition_key(path):
            singles.append((path, size, fmt))
        else:
            keyed.setdefault((fmt, partition_key(path)), []).append((path, size))
    groups: list[dict[str, Any]] = []
    left_out = 0
    for (fmt, _), members in keyed.items():
        if len(members) == 1:
            singles.append((members[0][0], members[0][1], fmt))
            continue
        kept = members[:MAX_PARTS]
        left_out += len(members) - len(kept)
        groups.append(
            {"paths": [p for p, _ in kept], "format": fmt, "size_bytes": sum(s for _, s in kept)}
        )
    groups += [{"paths": [p], "format": f, "size_bytes": s} for p, s, f in singles]
    groups.sort(key=lambda g: str(g["paths"][0]))
    return groups, left_out


def _clean(text: str) -> str:
    return re.sub("[^a-z0-9]+", "_", text.lower()).strip("_")[:40] or "table"


def _stem(path: str) -> str:
    stem = PurePosixPath(path).name
    suffix, compression = suffixes(path)
    if compression:
        stem = stem[: -len(compression)]
    return stem[: -len(suffix)] if suffix and stem.lower().endswith(suffix) else stem


def group_names(groups: list[dict[str, Any]], prefix: str = "") -> list[str]:
    """A short name for each group: the file name when it is the only one called that.

    Partitions are named for what they share (``prices`` for ``2022/prices.csv`` and
    ``2023/prices.csv``). Tables that share a name are told apart by their format when that is
    enough (``load_csv``, ``load_parquet``) and by their folders when it is not.
    """
    entries: list[tuple[str, str]] = []
    for group in groups:
        paths = group["paths"]
        first = str(paths[0])
        shared = partition_key(first).replace("#", "") if len(paths) > 1 else first
        entries.append((shared, str(group["format"])))
    plain = [_clean(_stem(path)) for path, _ in entries]
    names: list[str] = []
    taken: set[str] = set()
    for (path, fmt), base in zip(entries, plain, strict=True):
        same = [f for (_, f), b in zip(entries, plain, strict=True) if b == base]
        if len(same) == 1:
            name = base
        elif len(set(same)) == len(same):
            name = base[:34] + "_" + fmt
        else:
            name = _clean(str(PurePosixPath(path).with_suffix("")))
        stem, count = name, 2
        while name in taken:
            name = f"{stem[:36]}_{count}"
            count += 1
        taken.add(name)
        names.append(prefix + name)
    return names


#: Formats the runner knows by name and does not open, so the Tool tab can say what a file is and
#: why it is left alone. Prose, images and the like are not listed: they are not data.
_UNSUPPORTED: dict[tuple[str, ...], tuple[str, str]] = {
    (".pkl", ".pickle", ".joblib"): (
        "pickle",
        "a pickle runs code when it is loaded, so it is never read",
    ),
    (".rds", ".rdata", ".rda"): ("R data", "R data files are not read"),
    (".mat",): ("MATLAB", "MATLAB files are not read"),
    (".h5", ".hdf5", ".he5"): ("HDF5", "HDF5 files that are not NetCDF are not read yet"),
    (".grib", ".grib2", ".grb", ".grb2"): (
        "GRIB",
        "GRIB needs a decoder that Colab does not ship",
    ),
    (".tif", ".tiff"): ("GeoTIFF", "raster images are not read yet"),
    (".shp", ".gpkg", ".geojson"): ("geospatial", "geospatial vector files are not read yet"),
    (".npy", ".npz"): ("NumPy", "NumPy arrays are not read"),
    (".feather", ".arrow"): ("Arrow", "Arrow files are not read yet"),
    (".ods",): ("OpenDocument spreadsheet", "only Excel workbooks are read"),
    (".xlsm", ".xlsb"): ("Excel with macros", "workbooks that can hold macros are never opened"),
    (".7z", ".rar", ".tar", ".tgz"): ("archive", "only zip archives are opened"),
}
_UNSUPPORTED_BY_SUFFIX = {
    suffix: found for suffixes_, found in _UNSUPPORTED.items() for suffix in suffixes_
}


def is_prose_or_media(path: str) -> bool:
    return suffixes(path)[0] in PROSE_SUFFIXES


def unsupported_format(path: str) -> tuple[str, str] | None:
    """``(name, reason)`` for a data file the runner recognises and leaves alone, else None."""
    if is_about_the_primitive(path):
        return None
    name = PurePosixPath(path).name.lower()
    if name.endswith((".tar.gz", ".tar.bz2", ".tar.xz")):
        return _UNSUPPORTED_BY_SUFFIX[".tar"]
    suffix, _ = suffixes(path)
    return _UNSUPPORTED_BY_SUFFIX.get(suffix)
''')
Path(WORKDIR, "powerai_dataset_runner.py").write_text('''"""PowerAI dataset runner: makes a dataset of tabular files queryable over the PowerAI contract.

Reviewed and owned by the Hub. The launcher notebook writes this file next to an unpacked
dataset that has no code of its own, and serves it. It reads ``powerai_dataset.json`` from the
same folder: the list of tables (a name, a path inside the primitive, a format).

It only reads. Nothing is executed from a file: no pickle, Excel is read as values (formulas are
not evaluated, macros are not run), and a file over its size cap is never read whole. Rows
returned are capped, and so is every list.

Tools (POST /run, ``{"powerai": "1", "tool": ..., "input": {...}}``):

- describe:  tables, their columns and types, row count, time range
- head:      the first rows of a table
- query:     columns, filters and a time range, rows capped at 1,000
- aggregate: mean, sum, min, max or count, by columns and/or a time step (resample)
- stats:     one column: min, max, mean, quantiles, nulls (or top values for text)

GET /health does no I/O. Timestamps come back as ISO 8601 with their offset when the file has
one; a file whose times carry no time zone says so, and nothing is guessed. NaN is null.
"""

from __future__ import annotations

import bz2
import contextlib
import gzip
import hashlib
import json
import lzma
import math
import re
import sqlite3
import sys
import time
import zipfile
from functools import lru_cache
from pathlib import Path, PurePosixPath
from typing import Any
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse

HERE = Path(__file__).resolve().parent
# powerai_dataset_files.py is written beside this file: the rules for what a table is.
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))
from powerai_dataset_files import (  # noqa: E402
    CONTAINERS,
    MAX_PARTS,
    group_files,
    group_names,
    table_format,
)

CONFIG_FILENAME = "powerai_dataset.json"
TOOLS = ("describe", "head", "query", "aggregate", "stats", "slice")

MAX_ROWS = 1000
MAX_HEAD = 100
MAX_LIST = 50
SAMPLE_ROWS = 1000
MAX_TABLES_DESCRIBED = 10
MAX_FILTERS = 10
MAX_MEASURES = 10
MAX_GROUP_BY = 5
MAX_PARQUET_CELLS = 60_000_000
#: A SQLite table is read whole into memory, so it is capped by what it holds, not by file size.
MAX_SQLITE_CELLS = 20_000_000
MAX_SQLITE_TABLES = 200
SQLITE_SECONDS = 20
#: ``slice`` returns at most this many values, and reads at most this many to work out statistics.
MAX_CELLS = 10_000
MAX_STAT_CELLS = 50_000_000
GRID_ATTRIBUTES = 15

TAB = chr(9)
MB = 1024 * 1024
#: The most a file may weigh on disk to be read whole, by format. Compressed files are checked by
#: their compressed size, which is why theirs is lower. Parquet is not read whole: only the
#: columns a call needs are read, and that is capped by cells.
SCAN_CAP = {
    "csv": 250 * MB,
    "tsv": 250 * MB,
    "csv.gz": 60 * MB,
    "json": 100 * MB,
    "jsonl": 100 * MB,
    "xlsx": 30 * MB,
    "xls": 30 * MB,
}
FORMATS = ("csv", "tsv", "json", "jsonl", "xlsx", "xls", "parquet", "zip", "sqlite", "netcdf")
COMPRESSION = {".gz": "gzip", ".bz2": "bz2", ".xz": "xz"}

QUANTILES = (0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99)
AGGS = ("mean", "sum", "min", "max", "count")
FILTER_OPS = ("eq", "ne", "in", "gt", "gte", "lt", "lte")
# English, plus the German words that name time columns in Central European exports.
TIME_WORDS = ("timestamp", "datetime", "date", "time", "datum", "zeit")
DAY_FIRST = re.compile("^([0-9]{1,2})([./-])([0-9]{1,2})[./-][0-9]{4}")
HEADER_SCAN_ROWS = 20
ISO_TIME = re.compile("^[0-9]{4}-[0-9]{2}-[0-9]{2}[ T][0-9]{2}:[0-9]{2}")
RESAMPLE = re.compile("^[0-9]{1,3}(min|h|D|W|MS|ME|QS|QE|YS|YE)$")
OFFSET_END = re.compile("(Z|[+-][0-9][0-9]:?[0-9][0-9])$")

app = FastAPI(title="PowerAI dataset runner")


class _InputError(Exception):
    """The caller asked for something the tool cannot do: said in the reply."""


class _DataError(Exception):
    """The file could not be read: said in the reply, without host detail."""


# --- the tables --------------------------------------------------------------------------

SOURCE_COLUMN = "source_file"
ZIP_MAX_MEMBERS = 2000
ZIP_MAX_RATIO = 1000
ZIP_PARQUET_CAP = 100 * MB
CACHE_DIR = HERE / ".powerai_cache"
EXTRACTED_KEPT = 6

#: What describe says about a zip it opened, and about partitions whose columns differ. Filled as
#: the tables are listed and read.
_NOTES: dict[str, str] = {}
_PARTIAL: dict[str, list[str]] = {}
_EXTRACTED: dict[str, Path] = {}


def _inside(relative: str) -> Path | None:
    path = (HERE / relative).resolve()
    return path if path.is_relative_to(HERE) and path.is_file() else None


def _is_record_list(head: bytes) -> bool:
    return head.decode("utf-8-sig", errors="replace").lstrip()[:1] == "["


def _zip_tables(name: str, relative: str) -> list[dict[str, Any]]:
    """The tables inside a zip: its readable members, partitions grouped, nested zips left alone.

    Only the zip's list of members is read here. A member is unpacked, to a file of its own and
    under a size limit, when a call needs it.
    """
    path = _inside(relative)
    if path is None:
        _NOTES[name] = "The zip file is not in the primitive."
        return []
    members: list[tuple[str, int]] = []
    nested = suspicious = 0
    try:
        with zipfile.ZipFile(path) as archive:
            infos = [i for i in archive.infolist() if not i.is_dir()]
            for info in infos[:ZIP_MAX_MEMBERS]:
                fmt = table_format(info.filename)
                if fmt in CONTAINERS:
                    nested += 1
                    continue
                if fmt is None or not info.file_size:
                    continue
                if info.compress_size and info.file_size / info.compress_size > ZIP_MAX_RATIO:
                    suspicious += 1
                    continue
                if fmt == "json":
                    with archive.open(info) as handle:
                        if not _is_record_list(handle.read(256)):
                            continue
                members.append((info.filename, info.file_size))
    except (zipfile.BadZipFile, OSError):
        _NOTES[name] = "Could not be opened as a zip."
        return []
    sizes = dict(members)
    groups, left_out = group_files(members)
    notes = []
    if nested:
        noun = (
            "zip, database or gridded file" if nested == 1 else "zips, databases or gridded files"
        )
        notes.append(str(nested) + " " + noun + " inside it, not opened")
    if suspicious:
        notes.append(str(suspicious) + " not read (they expand far more than a data file does)")
    if left_out:
        notes.append(str(left_out) + " files left out: a table has at most " + str(MAX_PARTS))
    if len(infos) > ZIP_MAX_MEMBERS:
        notes.append("only the first " + str(ZIP_MAX_MEMBERS) + " files were listed")
    if not groups:
        notes.append("no file in it is a format the runner reads")
    if notes:
        _NOTES[name] = "; ".join(notes) + "."
    tables = []
    for table_name, group in zip(group_names(groups, prefix=name + "."), groups, strict=True):
        parts = [
            {"zip": relative, "member": m, "label": relative + "!" + m, "size": sizes[m]}
            for m in group["paths"]
        ]
        tables.append({"name": table_name, "format": group["format"], "parts": parts})
    return tables


def _slug(text: str) -> str:
    return re.sub("[^a-z0-9]+", "_", text.lower()).strip("_")[:40] or "table"


def _quote(name: str) -> str:
    """A name from the database's own list, as a quoted SQL identifier. Never a caller's text."""
    return chr(34) + name.replace(chr(34), chr(34) * 2) + chr(34)


def _sqlite_open(path: Path) -> sqlite3.Connection:
    """A SQLite database opened read-only, with a time budget and without trusting its schema.

    Nothing a caller sends is ever put in a query: table names come from the database's own list
    and are quoted. ``mode=ro`` and ``query_only`` mean nothing can be written, ``trusted_schema``
    off stops a view or trigger from calling functions that are not safe, and the progress handler
    abandons a query that runs past ``SQLITE_SECONDS``, so a hostile view cannot hang the session.
    """
    with open(path, "rb") as handle:
        head = handle.read(16)
    if head != b"SQLite format 3" + bytes([0]):
        raise _DataError("This file is not a SQLite database.")
    connection = sqlite3.connect(path.as_uri() + "?mode=ro", uri=True)
    connection.execute("PRAGMA query_only = ON")
    connection.execute("PRAGMA trusted_schema = OFF")
    started = time.monotonic()
    connection.set_progress_handler(
        lambda: int(time.monotonic() - started > SQLITE_SECONDS), 100_000
    )
    return connection


def _sqlite_tables(name: str, relative: str) -> list[dict[str, Any]]:
    """The tables and views of a SQLite database, each one a table named ``<file>.<table>``."""
    path = _inside(relative)
    if path is None:
        _NOTES[name] = "The database is not in the primitive."
        return []
    try:
        connection = _sqlite_open(path)
        try:
            listed = connection.execute(
                "SELECT name FROM sqlite_master WHERE type IN ('table', 'view') "
                "AND name NOT LIKE 'sqlite_%' ORDER BY name COLLATE NOCASE"
            ).fetchall()
        finally:
            connection.close()
    except (sqlite3.Error, _DataError, OSError):
        _NOTES[name] = "Could not be opened as a SQLite database."
        return []
    if not listed:
        _NOTES[name] = "The database holds no tables."
        return []
    notes = []
    if len(listed) > MAX_SQLITE_TABLES:
        notes.append(
            "only the first "
            + str(MAX_SQLITE_TABLES)
            + " of "
            + str(len(listed))
            + " tables are listed"
        )
    taken: set[str] = set()
    tables = []
    for (inner,) in listed[:MAX_SQLITE_TABLES]:
        label = name + "." + _slug(str(inner))
        base, count = label, 2
        while label in taken:
            label = base + "_" + str(count)
            count += 1
        taken.add(label)
        part = {"path": relative, "label": relative + "#" + str(inner), "sqlite_table": str(inner)}
        tables.append({"name": label, "format": "sqlite", "parts": [part]})
    if notes:
        _NOTES[name] = "; ".join(notes) + "."
    return tables


@lru_cache(maxsize=16)
def _sqlite_shape(path: str, inner: str) -> tuple[int, int]:
    """Rows and columns of one SQLite table, counted by the database."""
    connection = _sqlite_open(Path(path))
    try:
        rows = int(connection.execute("SELECT COUNT(*) FROM " + _quote(inner)).fetchone()[0])
        cursor = connection.execute("SELECT * FROM " + _quote(inner) + " LIMIT 0")
        return rows, len(cursor.description or [])
    except sqlite3.Error:
        raise _DataError(
            "The SQLite table could not be counted: it may take too long to read."
        ) from None
    finally:
        connection.close()


def _read_sqlite(path: Path, inner: str, nrows: int | None) -> pd.DataFrame:
    connection = _sqlite_open(path)
    try:
        sql = "SELECT * FROM " + _quote(inner)
        if nrows is not None:
            sql += " LIMIT " + str(int(nrows))
        return pd.read_sql_query(sql, connection)
    except Exception:  # noqa: BLE001 - pandas wraps the database's own errors
        raise _DataError(
            "The SQLite table could not be read: it may be damaged, or take too long."
        ) from None
    finally:
        connection.close()


@lru_cache(maxsize=1)
def _tables() -> dict[str, dict[str, Any]]:
    config = json.loads((HERE / CONFIG_FILENAME).read_text())
    if not isinstance(config, dict) or not isinstance(config.get("tables"), list):
        raise _DataError(CONFIG_FILENAME + " must hold a list of tables.")
    tables: dict[str, dict[str, Any]] = {}
    for entry in config["tables"]:
        if not isinstance(entry, dict) or entry.get("format") not in FORMATS:
            continue
        name = str(entry.get("name"))
        listed = entry.get("paths") if isinstance(entry.get("paths"), list) else [entry.get("path")]
        paths = [str(p) for p in listed if p][:MAX_PARTS]
        if not paths:
            continue
        if entry["format"] == "zip":
            for table in _zip_tables(name, paths[0]):
                tables[table["name"]] = table
        elif entry["format"] == "sqlite":
            for table in _sqlite_tables(name, paths[0]):
                tables[table["name"]] = table
        else:
            parts = [{"path": p, "label": p} for p in paths]
            tables[name] = {"name": name, "format": entry["format"], "parts": parts}
    return tables


def _path(part: dict[str, Any]) -> Path:
    if part.get("zip"):
        return _extract(part)
    path = _inside(part["path"])
    if path is None:
        raise _DataError("A file for this table is not in the primitive.")
    return path


def _extract(part: dict[str, Any]) -> Path:
    """A zip member as a file of its own, under the size limit for its format.

    The file is named by the member's last name only, never by a path from the zip, so nothing
    the zip says can place a file anywhere else. A few are kept; older ones are deleted.
    """
    key = part["label"]
    kept = _EXTRACTED.get(key)
    if kept is not None and kept.is_file():
        return kept
    archive_path = _inside(part["zip"])
    if archive_path is None:
        raise _DataError("The zip file is not in the primitive.")
    member = str(part["member"])
    cap = _cap_for(table_format(member) or "csv", member, True)
    folder = CACHE_DIR / hashlib.sha1(key.encode()).hexdigest()[:16]
    folder.mkdir(parents=True, exist_ok=True)
    target = folder / PurePosixPath(member).name
    written = 0
    try:
        with (
            zipfile.ZipFile(archive_path) as archive,
            archive.open(member) as source,
            open(target, "wb") as out,
        ):
            while True:
                chunk = source.read(MB)
                if not chunk:
                    break
                written += len(chunk)
                if cap is not None and written > cap:
                    raise _InputError(member + " is larger than the limit for a file inside a zip.")
                out.write(chunk)
    except _InputError:
        target.unlink(missing_ok=True)
        raise
    except (zipfile.BadZipFile, KeyError, OSError):
        target.unlink(missing_ok=True)
        raise _DataError("A file inside the zip could not be read.") from None
    _EXTRACTED[key] = target
    while len(_EXTRACTED) > EXTRACTED_KEPT:
        old = _EXTRACTED.pop(next(iter(_EXTRACTED)))
        old.unlink(missing_ok=True)
        with contextlib.suppress(OSError):
            old.parent.rmdir()
    return target


def _pick(inp: dict[str, Any]) -> dict[str, Any]:
    tables = _tables()
    name = inp.get("table")
    if name is None and len(tables) == 1:
        return next(iter(tables.values()))
    if not isinstance(name, str) or name not in tables:
        names = ", ".join(list(tables)[:MAX_LIST])
        raise _InputError("table must be one of: " + names + ".")
    return tables[name]


def _compression(name: str) -> str | None:
    return COMPRESSION.get(PurePosixPath(name).suffix.lower())


def _cap_for(fmt: str, name: str, in_zip: bool) -> int | None:
    """The most bytes a file of this format may weigh to be read whole (None: no such limit).

    A parquet file is read by column, a SQLite table is capped by its cells and a NetCDF file is
    read by selection, so none of them has a limit on its size on disk.
    """
    if fmt in ("sqlite", "netcdf"):
        return None
    if fmt == "parquet":
        return ZIP_PARQUET_CAP if in_zip else None
    if _compression(name) and fmt in ("csv", "tsv"):
        return SCAN_CAP["csv.gz"]
    return SCAN_CAP[fmt]


def _part_name(part: dict[str, Any]) -> str:
    return str(part.get("member") or part["path"])


def _table_size(table: dict[str, Any]) -> int:
    return sum(int(p.get("size") or _path(p).stat().st_size) for p in table["parts"])


def _too_big(table: dict[str, Any]) -> bool:
    if table["format"] == "sqlite":
        rows, columns = _sqlite_shape(
            str(_path(table["parts"][0])), table["parts"][0]["sqlite_table"]
        )
        return rows * columns > MAX_SQLITE_CELLS
    first = table["parts"][0]
    cap = _cap_for(table["format"], _part_name(first), bool(first.get("zip")))
    return cap is not None and _table_size(table) > cap


def _first_lines(path: Path, count: int) -> list[str]:
    opener: Any = {".gz": gzip.open, ".bz2": bz2.open, ".xz": lzma.open}.get(
        path.suffix.lower(), open
    )
    with opener(path, mode="rt", encoding="utf-8", errors="replace") as handle:
        return [handle.readline() for _ in range(count)]


def _first_char(path: Path) -> str:
    with open(path, encoding="utf-8", errors="replace") as handle:
        return handle.read(256).lstrip().lstrip(chr(65279))[:1]


def _delimited_options(fmt: str, path: Path) -> dict[str, Any]:
    """Separator and decimal mark. Most files are as named; European exports use ; and 1,5."""
    if fmt == "tsv":
        return {"sep": TAB}
    lines = _first_lines(path, 2)
    head = lines[0]
    sep = max((",", ";", "|", TAB), key=head.count)
    options: dict[str, Any] = {"sep": sep if head.count(sep) else ","}
    if sep == ";" and re.search("[0-9],[0-9]", lines[1] or ""):
        options["decimal"] = ","
    return options


# --- reading -----------------------------------------------------------------------------


def _read_part(
    table: dict[str, Any],
    part: dict[str, Any],
    sheet: str | None = None,
    columns: list[str] | None = None,
    nrows: int | None = None,
) -> pd.DataFrame:
    """One file of a table as a frame. ``nrows`` and ``columns`` are only ever a cut."""
    path = _path(part)
    fmt = table["format"]
    try:
        if fmt in ("csv", "tsv"):
            return pd.read_csv(
                path,
                usecols=columns,
                nrows=nrows,
                low_memory=False,
                **_delimited_options(fmt, path),
            )
        if fmt in ("xlsx", "xls"):
            # Values only: openpyxl is opened read-only with cached values, so no formula is
            # evaluated and no macro is run. xlsm and xlsb are not offered at all.
            return pd.read_excel(
                path,
                sheet_name=sheet if sheet is not None else 0,
                engine="openpyxl" if fmt == "xlsx" else "xlrd",
                header=_excel_header(str(path), fmt, sheet),
                usecols=columns,
                nrows=nrows,
            )
        if fmt == "sqlite":
            return _read_sqlite(path, part["sqlite_table"], nrows)
        if fmt == "jsonl":
            return pd.read_json(path, lines=True, nrows=nrows)
        if fmt == "json":
            if _first_char(path) != "[":
                raise _DataError(
                    "A .json file must be a list of records, like [{...}, {...}]. This one is not."
                )
            frame = pd.read_json(path, orient="records")
            return frame[columns] if columns else frame
        import pyarrow.parquet as pq

        file = pq.ParquetFile(path)
        # A partition may lack a column the others have: read what it has.
        have = set(file.schema_arrow.names)
        use = [c for c in columns if c in have] if columns else None
        if nrows is not None:
            batch = next(file.iter_batches(batch_size=nrows, columns=use), None)
            return (
                batch.to_pandas()
                if batch is not None
                else file.schema_arrow.empty_table().to_pandas()
            )
        return file.read(columns=use).to_pandas()
    except (_InputError, _DataError):
        raise
    except KeyError as exc:
        raise _InputError("No such sheet or column: " + str(exc).strip("'") + ".") from None
    except ValueError as exc:
        raise _DataError("The file could not be read as " + fmt + ": " + str(exc)[:200]) from None
    except Exception:  # noqa: BLE001 - any parser failure is the same answer to the caller
        raise _DataError("The file could not be read as " + fmt + ".") from None


def _read(
    table: dict[str, Any],
    sheet: str | None = None,
    columns: list[str] | None = None,
    nrows: int | None = None,
) -> pd.DataFrame:
    """A table as a frame: one file, or its partitions stacked with a column naming each file."""
    parts = table["parts"]
    if len(parts) == 1:
        return _read_part(table, parts[0], sheet, columns, nrows)
    wanted = [c for c in columns if c != SOURCE_COLUMN] if columns else None
    frames: list[pd.DataFrame] = []
    rows = 0
    for part in parts:
        left = None if nrows is None else nrows - rows
        frame = _read_part(table, part, sheet, wanted or None, left)
        label = SOURCE_COLUMN if SOURCE_COLUMN not in frame.columns else SOURCE_COLUMN + "_"
        frame.insert(0, label, part["label"])
        frames.append(frame)
        rows += len(frame)
        if nrows is not None and rows >= nrows:
            break
    stacked = pd.concat(frames, ignore_index=True)
    missing = [c for c in stacked.columns if any(c not in f.columns for f in frames)]
    if missing:
        _PARTIAL[table["name"]] = [str(c) for c in missing][:MAX_LIST]
    return stacked[[c for c in columns if c in stacked.columns]] if columns else stacked


@lru_cache(maxsize=16)
def _excel_header(path: str, fmt: str, sheet: str | None) -> int:
    """The row (from 0) holding the column names: a title or blank rows may sit above it.

    It is the first of the top rows that is all text and has about as many cells as the widest
    one. A sheet whose first row is already that gets 0, which is what pandas assumes.
    """
    try:
        raw = pd.read_excel(
            path,
            sheet_name=sheet if sheet is not None else 0,
            engine="openpyxl" if fmt == "xlsx" else "xlrd",
            header=None,
            nrows=HEADER_SCAN_ROWS,
        )
    except Exception:  # noqa: BLE001 - a sheet that cannot be scanned is read as pandas would
        return 0
    widest = int(raw.notna().sum(axis=1).max()) if len(raw) else 0
    for index in range(len(raw)):
        cells = raw.iloc[index].dropna()
        if len(cells) >= max(2, int(widest * 0.6)) and all(isinstance(v, str) for v in cells):
            return index
    return 0


def _sheets(table: dict[str, Any]) -> list[str] | None:
    if table["format"] not in ("xlsx", "xls"):
        return None
    try:
        engine = "openpyxl" if table["format"] == "xlsx" else "xlrd"
        path = _path(table["parts"][0])
        return [str(s) for s in pd.ExcelFile(path, engine=engine).sheet_names][:MAX_LIST]
    except _DataError:
        raise
    except Exception:  # noqa: BLE001
        raise _DataError("The workbook could not be opened.") from None


@lru_cache(maxsize=8)
def _parquet_scan(name: str) -> tuple[pd.DataFrame, int]:
    """An empty frame with every column of every file (types included), and the rows of all.

    Only the files' footers are read, so this is cheap however large the files are. The columns
    are the union: a partition may have one the others lack, and those are recorded.
    """
    import pyarrow.parquet as pq

    table = _tables()[name]
    schemas: list[dict[str, Any]] = []
    rows = 0
    for part in table["parts"]:
        file = pq.ParquetFile(_path(part))
        empty = file.schema_arrow.empty_table().to_pandas()
        schemas.append({str(c): empty[c].dtype for c in empty.columns})
        rows += int(file.metadata.num_rows)
    dtypes: dict[str, Any] = {}
    for schema in schemas:
        for column, dtype in schema.items():
            dtypes.setdefault(column, dtype)
    if len(schemas) > 1:
        missing = [c for c in dtypes if any(c not in schema for schema in schemas)]
        if missing:
            _PARTIAL[name] = missing[:MAX_LIST]
        dtypes = {SOURCE_COLUMN: pd.StringDtype(), **dtypes}
    return pd.DataFrame({c: pd.Series(dtype=d) for c, d in dtypes.items()}), rows


def _parquet_meta(table: dict[str, Any]) -> tuple[list[str], int]:
    """Every column name (the source column first, for partitions) and the rows of every file."""
    union, rows = _parquet_scan(table["name"])
    return [str(c) for c in union.columns], rows


@lru_cache(maxsize=2)
def _whole(name: str, sheet: str | None) -> pd.DataFrame:
    """A table read whole, once, with its time column made a time. Small files only."""
    table = _tables()[name]
    if _too_big(table):
        raise _InputError(
            "Table "
            + name
            + " is too large to scan ("
            + _size_phrase(table)
            + "). Use describe or head, which read only the start of it."
        )
    return _coerce_times(_read(table, sheet))


def _megabytes(size: int) -> str:
    return str(round(size / MB)) + " MB"


def _size_phrase(table: dict[str, Any]) -> str:
    """How big a table that is too big is: its rows and columns for a database, else its size."""
    if table["format"] == "sqlite":
        part = table["parts"][0]
        rows, columns = _sqlite_shape(str(_path(part)), part["sqlite_table"])
        return format(rows, ",") + " rows by " + str(columns) + " columns"
    return _megabytes(_table_size(table))


def _frame(table: dict[str, Any], sheet: str | None, needed: list[str]) -> pd.DataFrame:
    """The table with at least the ``needed`` columns. Parquet reads only those."""
    if table["format"] == "parquet":
        names, rows = _parquet_meta(table)
        for column in needed:
            if column not in names:
                raise _InputError("No column named " + column + " in table " + table["name"] + ".")
        read = list(dict.fromkeys(needed)) or names
        # Time columns must be there to be filtered on, so they are always read.
        time = _time_name_from_schema(table)
        if time and time not in read and needed:
            read.append(time)
        if rows * len(read) > MAX_PARQUET_CELLS:
            raise _InputError(
                "Table " + table["name"] + " is too large to read these columns whole: "
                "name fewer columns, or use describe, head or stats on one column."
            )
        return _coerce_times(_read(table, None, read))
    return _whole(table["name"], sheet)


# --- times -------------------------------------------------------------------------------


def _is_time(series: pd.Series) -> bool:
    return pd.api.types.is_datetime64_any_dtype(series)


def _is_text(series: pd.Series) -> bool:
    """Text, whether pandas holds it as object (2.x) or as its own string type (3.x)."""
    return pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series)


def _day_first(values: pd.Series) -> bool | None:
    """Is a column of dd.mm.yyyy or mm/dd/yyyy style dates day-first? None when it cannot be told.

    Dotted dates are day-first (the German and Central European way). With slashes or dashes it
    is settled only by a value whose first part is above 12 (day-first) or whose second part is
    (month-first); when every value fits both, nothing is guessed. ISO dates lead with the year,
    match nothing here, and are never day-first.
    """
    month_first = False
    matched = False
    for value in values:
        found = DAY_FIRST.match(value)
        if found is None:
            continue
        matched = True
        if found.group(2) == "." or int(found.group(1)) > 12:
            return True
        month_first = month_first or int(found.group(3)) > 12
    if not matched:
        return False
    return False if month_first else None


def _to_datetime(series: pd.Series) -> pd.Series | None:
    """Parse a text column as times, or None when it is not clearly one.

    A column that would only half parse stays text: a time column with holes in it would answer
    a range query wrongly without saying so.
    """
    raw = series.dropna()
    if raw.empty or not all(isinstance(v, str) for v in raw.head(1000)):
        return None
    present = raw.astype(str).str.strip()
    sample = present.head(1000)
    has_offset = bool(sample.map(lambda s: bool(OFFSET_END.search(s))).all())
    first = _day_first(sample)
    if first is None:
        return None
    try:
        parsed = pd.to_datetime(series, utc=has_offset, errors="coerce", dayfirst=first)
    except (ValueError, TypeError):
        return None
    if parsed.notna().sum() < 0.99 * len(present):
        return None
    return parsed


def _looks_like_time(name: str) -> bool:
    lowered = name.lower()
    return any(word in lowered for word in TIME_WORDS)


def _iso_valued(series: pd.Series) -> bool:
    """A text column whose first values are all full ISO 8601 date-times, whatever it is called."""
    sample = series.dropna().head(50)
    # Bytes (a database BLOB) and numbers are never a time: only real text is looked at.
    if sample.empty or not all(isinstance(v, str) for v in sample):
        return False
    return all(ISO_TIME.match(v.strip()) is not None for v in sample)


def _coerce_times(frame: pd.DataFrame) -> pd.DataFrame:
    """Make a text column that is a time into one, when it parses. Nothing else is touched.

    The first column named like a time (``timestamp``, ``date``, ``Datum``) is tried first. If no
    column is named like a time, the first whose values are all ISO 8601 date-times is used.
    """
    if any(_is_time(frame[c]) for c in frame.columns):
        return frame
    text = [c for c in frame.columns if _is_text(frame[c])]
    named = [c for c in text if _looks_like_time(str(c))]
    for column in [*named, *(c for c in text if c not in named and _iso_valued(frame[c]))]:
        parsed = _to_datetime(frame[column])
        if parsed is not None:
            frame = frame.copy()
            frame[column] = parsed
            break
    return frame


def _time_name_from_schema(table: dict[str, Any]) -> str | None:
    import pyarrow as pa
    import pyarrow.parquet as pq

    schema = pq.ParquetFile(_path(table["parts"][0])).schema_arrow
    for field in schema:
        if pa.types.is_timestamp(field.type) or pa.types.is_date(field.type):
            return str(field.name)
    return None


def _time_column(frame: pd.DataFrame, wanted: Any = None) -> str | None:
    if wanted is not None:
        if wanted not in frame.columns or not _is_time(frame[wanted]):
            raise _InputError(str(wanted) + " is not a column that reads as a time.")
        return str(wanted)
    for column in frame.columns:
        if _is_time(frame[column]):
            return str(column)
    return None


def _zone(frame: pd.DataFrame, column: str) -> str | None:
    tz = getattr(frame[column].dt, "tz", None)
    return str(tz) if tz is not None else None


def _with_time_zone(frame: pd.DataFrame, column: str, zone: Any) -> pd.DataFrame:
    """Show the time column in ``zone``: a file without a zone is read as being in it."""
    if zone is None:
        return frame
    try:
        info = ZoneInfo(str(zone))
    except Exception:  # noqa: BLE001
        raise _InputError(
            str(zone) + " is not a time zone name such as Australia/Sydney."
        ) from None
    frame = frame.copy()
    series = frame[column]
    if series.dt.tz is None:
        frame[column] = series.dt.tz_localize(info, ambiguous="NaT", nonexistent="NaT")
    else:
        frame[column] = series.dt.tz_convert(info)
    return frame


def _stamp(value: Any, frame: pd.DataFrame, column: str, what: str) -> pd.Timestamp:
    try:
        stamp = pd.Timestamp(str(value))
    except (ValueError, TypeError):
        raise _InputError(
            what + " must be an ISO 8601 time such as 2023-07-01T00:00:00Z."
        ) from None
    zone = frame[column].dt.tz
    if zone is None and stamp.tzinfo is not None:
        raise _InputError(
            "The times in "
            + column
            + " have no time zone, so "
            + what
            + " must be given without one."
        )
    if zone is not None and stamp.tzinfo is None:
        return stamp.tz_localize(zone)
    return stamp


# --- filtering ---------------------------------------------------------------------------


def _check_columns(frame: pd.DataFrame, names: list[str]) -> None:
    for name in names:
        if name not in frame.columns:
            raise _InputError("No column named " + name + ".")


def _value(series: pd.Series, value: Any) -> Any:
    if _is_time(series):
        return pd.Timestamp(str(value))
    if pd.api.types.is_numeric_dtype(series):
        if isinstance(value, bool) or not isinstance(value, int | float):
            raise _InputError("A number is needed to compare " + str(series.name) + ".")
        return value
    return str(value)


def _compare(series: pd.Series, op: str, value: Any) -> pd.Series:
    if op == "eq":
        return series == value
    if op == "ne":
        return series != value
    if op == "gt":
        return series > value
    if op == "gte":
        return series >= value
    if op == "lt":
        return series < value
    return series <= value


def _mask(frame: pd.DataFrame, inp: dict[str, Any]) -> tuple[pd.Series, str | None]:
    """Rows that pass the filters and the time range, and the time column's zone."""
    keep = pd.Series(True, index=frame.index)
    filters = inp.get("filters") or []
    if not isinstance(filters, list) or len(filters) > MAX_FILTERS:
        raise _InputError("filters must be a list of at most " + str(MAX_FILTERS) + ".")
    for item in filters:
        if not isinstance(item, dict) or item.get("op") not in FILTER_OPS:
            raise _InputError(
                "Each filter needs column, op (" + ", ".join(FILTER_OPS) + ") and value."
            )
        column = str(item.get("column"))
        _check_columns(frame, [column])
        series = frame[column]
        op, value = item["op"], item.get("value")
        if op == "in":
            if not isinstance(value, list) or not value or len(value) > MAX_LIST:
                raise _InputError("value for in must be a list of 1 to " + str(MAX_LIST) + ".")
            keep &= series.isin([_value(series, v) for v in value])
            continue
        keep &= _compare(series, op, _value(series, value))
    zone = None
    if inp.get("time_from") is not None or inp.get("time_to") is not None:
        column = _time_column(frame, inp.get("time_column"))
        if column is None:
            raise _InputError("This table has no time column. Name one with time_column.")
        zone = _zone(frame, column)
        if inp.get("time_from") is not None:
            keep &= frame[column] >= _stamp(inp["time_from"], frame, column, "time_from")
        if inp.get("time_to") is not None:
            keep &= frame[column] < _stamp(inp["time_to"], frame, column, "time_to")
    return keep, zone


def _prepared(table: dict[str, Any], inp: dict[str, Any], needed: list[str]) -> pd.DataFrame:
    """The frame for a call: read, its time column shown in ``time_zone`` when one was asked."""
    sheet = inp.get("sheet")
    if sheet is not None and not isinstance(sheet, str):
        raise _InputError("sheet must be a sheet name.")
    filtered = [str(f.get("column")) for f in (inp.get("filters") or []) if isinstance(f, dict)]
    wanted = inp.get("time_column")
    if wanted is not None and not isinstance(wanted, str):
        raise _InputError("time_column must be a column name.")
    frame = _frame(table, sheet, needed + filtered + ([wanted] if wanted else []))
    if wanted:
        if wanted not in frame.columns:
            raise _InputError("No column named " + wanted + " to use as the time.")
        if not _is_time(frame[wanted]):
            parsed = _to_datetime(frame[wanted])
            if parsed is None:
                raise _InputError(wanted + " does not read as a time.")
            frame = frame.copy()
            frame[wanted] = parsed
    zone = inp.get("time_zone")
    if zone is not None:
        column = _time_column(frame, inp.get("time_column"))
        if column is None:
            raise _InputError("This table has no time column to put in " + str(zone) + ".")
        frame = _with_time_zone(frame, column, zone)
    return frame


# --- output ------------------------------------------------------------------------------


def _plain(value: Any) -> Any:
    """Plain JSON data only: NaN, NaT and infinity become null; times are ISO 8601."""
    if value is None or value is pd.NaT:
        return None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, pd.Timedelta):
        return str(value)
    if isinstance(value, bool | str):
        return value
    if hasattr(value, "item") and not isinstance(value, list | tuple | dict):
        value = value.item()
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, int):
        return value
    if isinstance(value, list | tuple):
        return [_plain(v) for v in value]
    return str(value)


def _rows(frame: pd.DataFrame) -> list[list[Any]]:
    return [[_plain(v) for v in row] for row in frame.itertuples(index=False, name=None)]


def _dtype(series: pd.Series) -> str:
    if pd.api.types.is_bool_dtype(series):
        return "boolean"
    if pd.api.types.is_integer_dtype(series):
        return "integer"
    if pd.api.types.is_numeric_dtype(series):
        return "number"
    if _is_time(series):
        return "datetime"
    return "string"


def _int(inp: dict[str, Any], key: str, default: int, low: int, high: int) -> int:
    value = inp.get(key, default)
    if isinstance(value, bool) or not isinstance(value, int) or not low <= value <= high:
        raise _InputError(
            key + " must be a whole number from " + str(low) + " to " + str(high) + "."
        )
    return value


def _names(inp: dict[str, Any], key: str, limit: int) -> list[str]:
    value = inp.get(key)
    if value is None:
        return []
    if (
        not isinstance(value, list)
        or len(value) > limit
        or not all(isinstance(v, str) for v in value)
    ):
        raise _InputError(key + " must be a list of at most " + str(limit) + " column names.")
    return value


# --- metadata ----------------------------------------------------------------------------

DESCRIPTOR_FILES = ("datapackage.json", "metadata.json")
README_FILES = ("README.md", "readme.md", "Readme.md")
MAX_META_BYTES = 5 * MB
NAME_HEADERS = ("column", "columns", "column name", "name", "variable", "field", "attribute")
DESCRIPTION_HEADERS = ("description", "meaning", "definition", "explanation")
UNIT_HEADERS = ("unit", "units", "unit of measure", "uom")
EMPTY_CELLS = ("", "-", "n/a", "na", "none", "null")


@lru_cache(maxsize=1)
def _descriptors() -> list[tuple[str, dict[str, Any]]]:
    """The resources of a Data Package or OEMetadata file at the top of the primitive.

    Both keep a list of resources, each with ``schema.fields`` that may carry a ``unit`` and a
    ``description``, so one reader serves both.
    """
    found: list[tuple[str, dict[str, Any]]] = []
    for filename in DESCRIPTOR_FILES:
        path = _inside(filename)
        if path is None or path.stat().st_size > MAX_META_BYTES:
            continue
        try:
            data = json.loads(path.read_text(encoding="utf-8"))
        except ValueError:
            continue
        resources = data.get("resources") if isinstance(data, dict) else None
        if isinstance(resources, list):
            found += [(filename, r) for r in resources if isinstance(r, dict)]
    return found


def _cell(text: str) -> str:
    return text.strip().strip("`*_ ")


def _readme_table(lines: list[str]) -> dict[str, dict[str, str]]:
    """Column names with their unit and description, from one markdown table (or nothing)."""
    rows = [[_cell(c) for c in line.strip().strip("|").split("|")] for line in lines]
    header = [c.lower() for c in rows[0]]
    dashes = rows[1]
    if not all(c and set(c) <= set("-: ") for c in dashes):
        return {}
    name = next((i for i, h in enumerate(header) if h in NAME_HEADERS), None)
    unit = next((i for i, h in enumerate(header) if h in UNIT_HEADERS), None)
    text = next((i for i, h in enumerate(header) if h in DESCRIPTION_HEADERS), None)
    if name is None or (unit is None and text is None):
        return {}
    out: dict[str, dict[str, str]] = {}
    for row in rows[2:]:
        if len(row) <= name or not row[name]:
            continue
        entry: dict[str, str] = {}
        for key, index in (("unit", unit), ("description", text)):
            if index is not None and len(row) > index and row[index].lower() not in EMPTY_CELLS:
                entry[key] = row[index][:300]
        if entry:
            out[row[name]] = entry
    return out


@lru_cache(maxsize=1)
def _readme_tables() -> list[dict[str, dict[str, str]]]:
    text = ""
    for filename in README_FILES:
        path = _inside(filename)
        if path is not None and path.stat().st_size <= MAX_META_BYTES:
            text = path.read_text(encoding="utf-8", errors="replace")
            break
    tables: list[dict[str, dict[str, str]]] = []
    block: list[str] = []
    for line in [*text.splitlines(), ""]:
        if line.strip().startswith("|"):
            block.append(line)
            continue
        if len(block) >= 3:
            parsed = _readme_table(block)
            if parsed:
                tables.append(parsed)
        block = []
    return tables


def _norm(path: str) -> str:
    path = path.lower()
    while path.startswith("./"):
        path = path[2:]
    return path


def _resource_for(table: dict[str, Any]) -> tuple[str, dict[str, Any]] | None:
    """The descriptor resource that describes this table: by path, then by name."""
    found = _descriptors()
    labels = {_norm(_part_name(p)) for p in table["parts"]}
    bases = {PurePosixPath(_part_name(p)).name.lower() for p in table["parts"]}
    names = {table["name"].lower(), table["name"].lower().rsplit(".", 1)[-1]}
    for source, resource in found:
        listed = (
            resource.get("paths")
            if isinstance(resource.get("paths"), list)
            else [resource.get("path")]
        )
        for entry in listed:
            if isinstance(entry, str) and (
                _norm(entry) in labels or PurePosixPath(entry).name.lower() in bases
            ):
                return source, resource
        if str(resource.get("name", "")).lower() in names:
            return source, resource
    if len(found) == 1 and len(_tables()) == 1:
        return found[0]
    return None


def _column_meta(
    table: dict[str, Any], columns: list[str]
) -> tuple[dict[str, dict[str, str]], dict[str, Any]]:
    """Units and descriptions for this table's columns, and where they came from.

    A Data Package or OEMetadata resource is read first, then README tables fill what it left
    out. A column no source speaks of has neither: nothing is guessed from its name.
    """
    info: dict[str, dict[str, str]] = {}
    sources: list[str] = []
    description: str | None = None
    match = _resource_for(table)
    if match is not None:
        source, resource = match
        schema = resource.get("schema")
        fields = schema.get("fields") if isinstance(schema, dict) else None
        used = False
        for field in fields if isinstance(fields, list) else []:
            if not isinstance(field, dict) or field.get("name") not in columns:
                continue
            entry = info.setdefault(str(field["name"]), {})
            unit = field.get("unit")
            text = field.get("description") or field.get("title")
            if isinstance(unit, str) and unit.strip():
                entry["unit"] = unit.strip()
                used = True
            if isinstance(text, str) and text.strip():
                entry["description"] = text.strip()[:300]
                used = True
        if used:
            sources.append(source)
        about = resource.get("description")
        description = about.strip()[:300] if isinstance(about, str) and about.strip() else None
    for mapping in _readme_tables():
        hits = [c for c in mapping if c in columns]
        added = False
        for column in hits:
            entry = info.setdefault(column, {})
            for key, value in mapping[column].items():
                if key not in entry:
                    entry[key] = value
                    added = True
        if added and "README.md" not in sources:
            sources.append("README.md")
    return {k: v for k, v in info.items() if v}, {"sources": sources, "description": description}


# --- gridded files (NetCDF) --------------------------------------------------------------


def _grid_engine(path: Path) -> str:
    """Classic NetCDF is read by scipy, NetCDF4 (which is HDF5) by h5netcdf: chosen by the header."""
    with open(path, "rb") as handle:
        head = handle.read(8)
    if head[:3] == b"CDF":
        return "scipy"
    if head[:4] == bytes([137]) + b"HDF":
        return "h5netcdf"
    raise _DataError("This file is not NetCDF.")


@lru_cache(maxsize=2)
def _open_grid(name: str) -> Any:
    """A NetCDF file opened lazily: only what a call selects is read from disk."""
    try:
        import xarray as xr
    except ImportError:
        raise _DataError(
            "Reading NetCDF needs xarray and h5netcdf, which this session does not have."
        ) from None
    path = _path(_tables()[name]["parts"][0])
    engine = _grid_engine(path)
    try:
        return xr.open_dataset(path, engine=engine)
    except (ValueError, TypeError, OverflowError):
        # A calendar or unit the decoder cannot read: open it with the times as plain numbers.
        try:
            dataset = xr.open_dataset(path, engine=engine, decode_times=False)
        except Exception:  # noqa: BLE001
            raise _DataError("The NetCDF file could not be opened.") from None
        _NOTES[name] = "its times could not be decoded and are shown as numbers."
        return dataset
    except Exception:  # noqa: BLE001 - any reader failure is the same answer to the caller
        raise _DataError("The NetCDF file could not be opened.") from None


def _find_axis(dataset: Any, kind: str) -> str | None:
    """The time, latitude or longitude coordinate: by its CF attributes first, then by its name."""
    names = {
        "time": ("time", "valid_time", "t"),
        "lat": ("lat", "latitude", "y"),
        "lon": ("lon", "longitude", "x"),
    }[kind]
    wanted = {"time": "T", "lat": "Y", "lon": "X"}[kind]
    standard_name = {"time": "time", "lat": "latitude", "lon": "longitude"}[kind]
    direction = {"lat": "north", "lon": "east"}.get(kind)
    for key, coord in dataset.coords.items():
        if coord.ndim != 1:
            continue
        attrs = coord.attrs
        units = str(attrs.get("units", "")).lower()
        found = (
            str(attrs.get("axis", "")).upper() == wanted
            or str(attrs.get("standard_name", "")).lower() == standard_name
            or (kind == "time" and np.issubdtype(coord.dtype, np.datetime64))
            or (direction is not None and units.startswith("degree") and direction in units)
        )
        if found:
            return str(key)
    for key, coord in dataset.coords.items():
        if coord.ndim == 1 and str(key).lower() in names:
            return str(key)
    return None


def _text(value: Any, limit: int = 200) -> str | None:
    if not isinstance(value, str | int | float) or isinstance(value, bool):
        return None
    return str(value).strip()[:limit] or None


def _coordinate_summary(coord: Any) -> dict[str, Any]:
    out: dict[str, Any] = {"size": int(coord.size), "dtype": str(coord.dtype)}
    units = _text(coord.attrs.get("units"))
    if units:
        out["units"] = units
    calendar = _text(coord.attrs.get("calendar"))
    if calendar:
        out["calendar"] = calendar
    is_time = np.issubdtype(coord.dtype, np.datetime64)
    if coord.size and (is_time or np.issubdtype(coord.dtype, np.number)):
        low, high = coord.values.min(), coord.values.max()
        if is_time:
            out["min"], out["max"] = _plain(pd.Timestamp(low)), _plain(pd.Timestamp(high))
        else:
            out["min"], out["max"] = _plain(low), _plain(high)
    return out


def _describe_grid(table: dict[str, Any]) -> dict[str, Any]:
    dataset = _open_grid(table["name"])
    axes = {kind: _find_axis(dataset, kind) for kind in ("time", "lat", "lon")}
    variables = []
    for key, var in list(dataset.data_vars.items())[:200]:
        entry: dict[str, Any] = {
            "name": str(key),
            "dims": [str(d) for d in var.dims],
            "shape": [int(n) for n in var.shape],
            "dtype": str(var.dtype),
        }
        for label, value in (
            ("units", var.attrs.get("units")),
            ("long_name", var.attrs.get("long_name") or var.attrs.get("standard_name")),
        ):
            if _text(value):
                entry[label] = _text(value)
        variables.append(entry)
    out: dict[str, Any] = {
        "table": table["name"],
        "path": table["parts"][0]["label"],
        "format": "netcdf",
        "kind": "grid",
        "files": 1,
        "size_bytes": _table_size(table),
        "dimensions": {str(d): int(n) for d, n in dataset.sizes.items()},
        "axes": axes,
        "coordinates": {
            str(k): _coordinate_summary(c)
            for k, c in list(dataset.coords.items())[:MAX_LIST]
            if c.ndim == 1
        },
        "variables": variables,
        "attributes": {
            str(k): _text(v) for k, v in list(dataset.attrs.items())[:GRID_ATTRIBUTES] if _text(v)
        },
        "time": None,
    }
    if axes["time"] is not None:
        summary = out["coordinates"].get(axes["time"], {})
        out["time"] = {
            "coordinate": axes["time"],
            "from": summary.get("min"),
            "to": summary.get("max"),
            "time_zone": None,
            "note": "NetCDF times carry no time zone; none is assumed.",
        }
    if axes["lat"] is None or axes["lon"] is None:
        curved = any(c.ndim == 2 for c in dataset.coords.values())
        out["note"] = (
            "The latitude and longitude are 2-D (a curved grid), so slice selects by position "
            "with index, and says where each cell is."
            if curved
            else "No latitude and longitude axis was found, so slice selects by position only."
        )
    return out


def _number(inp: dict[str, Any], key: str) -> float | None:
    value = inp.get(key)
    if value is None:
        return None
    if isinstance(value, bool) or not isinstance(value, int | float) or not math.isfinite(value):
        raise _InputError(key + " must be a finite number.")
    return float(value)


def _grid_time(value: Any, what: str) -> pd.Timestamp:
    """A time to match against the file's times: UTC, with no zone, since the file states none."""
    try:
        stamp = pd.Timestamp(str(value))
    except (ValueError, TypeError):
        raise _InputError(
            what + " must be an ISO 8601 time such as 2023-07-01T00:00:00Z."
        ) from None
    return stamp.tz_convert("UTC").tz_localize(None) if stamp.tzinfo is not None else stamp


def _fit_longitude(coord: Any, value: float) -> float:
    """The longitude in the file's own convention: 0 to 360, or -180 to 180."""
    if float(coord.values.min()) >= 0 and float(coord.values.max()) > 180:
        return value % 360
    return (value + 180) % 360 - 180


def _span(index: Any, low: float | None, high: float | None) -> slice | None:
    """A label range over a coordinate that may run up or down."""
    if low is None and high is None:
        return None
    return slice(high, low) if index.is_monotonic_decreasing else slice(low, high)


def _pick_index(data: Any, inp: dict[str, Any], variable: str, chosen: dict[str, Any]) -> Any:
    """Other dimensions (a level, an ensemble member), taken by position."""
    index = inp.get("index")
    if index is None:
        return data
    if not isinstance(index, dict):
        raise _InputError(
            "index must be an object such as {" + chr(34) + "level" + chr(34) + ": 0}."
        )
    for dim, position in index.items():
        if dim not in data.dims or isinstance(position, bool) or not isinstance(position, int):
            raise _InputError("index names a dimension of " + variable + " and a whole number.")
        if not -data.sizes[dim] <= position < data.sizes[dim]:
            raise _InputError("index " + str(dim) + " must be below " + str(data.sizes[dim]) + ".")
        data = data.isel({dim: position})
        chosen[str(dim)] = position
    return data


def _pick_time(
    dataset: Any, data: Any, name: str | None, inp: dict[str, Any], chosen: dict[str, Any]
) -> Any:
    given = [k for k in ("time", "time_from", "time_to") if inp.get(k) is not None]
    if not given:
        return data
    if name is None:
        raise _InputError("This variable has no time dimension to select on.")
    try:
        if inp.get("time") is not None:
            if inp.get("time_from") is not None or inp.get("time_to") is not None:
                raise _InputError("Give time, or time_from and time_to, not both.")
            data = data.sel({name: _grid_time(inp["time"], "time")}, method="nearest")
            chosen["time"] = _plain(pd.Timestamp(data[name].values))
            return data
        low = (
            _grid_time(inp["time_from"], "time_from") if inp.get("time_from") is not None else None
        )
        high = _grid_time(inp["time_to"], "time_to") if inp.get("time_to") is not None else None
        if low is not None and high is not None and low > high:
            raise _InputError("time_from must not be after time_to.")
        index = dataset.indexes[name]
        if not (index.is_monotonic_increasing or index.is_monotonic_decreasing):
            raise _InputError("The file's times are not in order, so a range cannot be taken.")
        data = data.sel({name: _span(index, low, high)})
        chosen["time_from"], chosen["time_to"] = _plain(low), _plain(high)
    except TypeError:
        raise _InputError(
            "This file's times use a calendar that cannot be selected by date."
        ) from None
    return data


def _pick_area(
    dataset: Any,
    data: Any,
    lat: str | None,
    lon: str | None,
    inp: dict[str, Any],
    chosen: dict[str, Any],
) -> Any:
    point = {k: _number(inp, k) for k in ("lat", "lon")}
    box = {k: _number(inp, k) for k in ("lat_min", "lat_max", "lon_min", "lon_max")}
    asked_point = any(v is not None for v in point.values())
    asked_box = any(v is not None for v in box.values())
    if (asked_point or asked_box) and lat is None and lon is None:
        if any(c.ndim == 2 for c in dataset.coords.values()):
            raise _InputError(
                "This file's latitude and longitude are 2-D (a curved grid), so an area cannot be "
                "chosen by them. Select by position with index, e.g. {"
                + chr(34)
                + "j"
                + chr(34)
                + ": 2, "
                + chr(34)
                + "i"
                + chr(34)
                + ": 4}; the reply "
                "gives the latitude and longitude of the cell."
            )
        raise _InputError("This variable has no latitude or longitude dimension to select on.")
    if asked_point and asked_box:
        raise _InputError(
            "Give a point (lat, lon) or a box (lat_min, lat_max, lon_min, lon_max), not both."
        )
    for kind, name in (("lat", lat), ("lon", lon)):
        if name is None:
            continue
        coord = dataset[name]
        fit = (lambda v: v) if kind == "lat" else (lambda v, c=coord: _fit_longitude(c, v))
        if point[kind] is not None:
            data = data.sel({name: fit(point[kind])}, method="nearest")
            chosen[kind] = _plain(data[name].values)
            continue
        low, high = box[kind + "_min"], box[kind + "_max"]
        if low is not None and high is not None and low > high:
            raise _InputError(kind + "_min must not be above " + kind + "_max.")
        if kind == "lon":
            low = None if low is None else fit(low)
            high = None if high is None else fit(high)
            if low is not None and high is not None and low > high:
                raise _InputError(
                    "That area crosses the longitude edge of the file. Ask for it in two parts."
                )
        span = _span(dataset.indexes[name], low, high)
        if span is not None:
            data = data.sel({name: span})
            chosen[kind + "_range"] = [box[kind + "_min"], box[kind + "_max"]]
    return data


def _select(dataset: Any, variable: str, inp: dict[str, Any]) -> tuple[Any, dict[str, Any]]:
    """The variable cut down to the positions, time and area asked for, and what that came to."""
    if variable not in dataset.data_vars:
        names = ", ".join(str(k) for k in list(dataset.data_vars)[:MAX_LIST])
        raise _InputError("variable must be one of: " + names + ".")
    data = dataset[variable]
    if not (np.issubdtype(data.dtype, np.number) or np.issubdtype(data.dtype, np.bool_)):
        raise _InputError(variable + " is not numeric, so it has no values to slice.")
    axes = {kind: _find_axis(dataset, kind) for kind in ("time", "lat", "lon")}
    chosen: dict[str, Any] = {}
    data = _pick_index(data, inp, variable, chosen)
    time_name = axes["time"] if axes["time"] in data.dims else None
    data = _pick_time(dataset, data, time_name, inp, chosen)
    lat_name = axes["lat"] if axes["lat"] in data.dims else None
    lon_name = axes["lon"] if axes["lon"] in data.dims else None
    data = _pick_area(dataset, data, lat_name, lon_name, inp, chosen)
    stride = inp.get("stride")
    if stride is not None:
        if isinstance(stride, bool) or not isinstance(stride, int) or not 1 <= stride <= 1000:
            raise _InputError("stride must be a whole number from 1 to 1000.")
        steps = {n: slice(None, None, stride) for n in (lat_name, lon_name) if n in data.dims}
        if steps:
            data = data.isel(steps)
            chosen["stride"] = stride
    return data, chosen


def _coordinate_values(data: Any, name: str) -> list[Any]:
    values = data[name].values
    if np.issubdtype(values.dtype, np.datetime64):
        return [_plain(pd.Timestamp(v)) for v in values]
    return [_plain(v) for v in values.tolist()]


def _grid_table(inp: dict[str, Any]) -> dict[str, Any]:
    tables = _tables()
    grids = [t for t in tables.values() if t["format"] == "netcdf"]
    name = inp.get("table")
    if name is None and len(grids) == 1:
        return grids[0]
    if isinstance(name, str) and name in tables and tables[name]["format"] == "netcdf":
        return tables[name]
    names = ", ".join(t["name"] for t in grids[:MAX_LIST]) or "none"
    raise _InputError("slice reads a NetCDF file. table must be one of: " + names + ".")


def slice_values(inp: dict[str, Any]) -> dict[str, Any]:
    """One variable of a NetCDF file over a time range and an area: its values, or its statistics."""
    table = _grid_table(inp)
    variable = inp.get("variable")
    if not isinstance(variable, str):
        raise _InputError("variable must be the name of a variable.")
    output = inp.get("output", "values")
    if output not in ("values", "stats"):
        raise _InputError("output must be values or stats.")

    dataset = _open_grid(table["name"])
    data, chosen = _select(dataset, variable, inp)
    cells = int(np.prod(data.shape)) if data.ndim else 1
    out: dict[str, Any] = {
        "variable": variable,
        "units": _text(data.attrs.get("units")),
        "long_name": _text(data.attrs.get("long_name") or data.attrs.get("standard_name")),
        "dims": [str(d) for d in data.dims],
        "shape": [int(n) for n in data.shape],
        "cells": cells,
        "selection": chosen,
        "time_zone": None,
    }
    if _find_axis(dataset, "time") is not None:
        out["time_note"] = (
            "NetCDF times carry no time zone. A time given with an offset was converted to UTC."
        )
    scalars = {
        str(k): _plain(pd.Timestamp(c.values) if c.dtype.kind == "M" else c.values.item())
        for k, c in data.coords.items()
        if c.ndim == 0
    }
    if scalars:
        out["at"] = scalars
    if output == "values":
        if cells > MAX_CELLS:
            raise _InputError(
                "The selection has "
                + format(cells, ",")
                + " values; at most "
                + format(MAX_CELLS, ",")
                + " are returned. Narrow the area or the times, raise "
                "stride, or ask for stats."
            )
        out["coords"] = {str(d): _coordinate_values(data, str(d)) for d in data.dims}
        out["values"] = _plain(data.values.tolist())
        return out
    if cells > MAX_STAT_CELLS:
        raise _InputError(
            "The selection has " + format(cells, ",") + " values, too many to read at once. "
            "Narrow the area or the times, or raise stride."
        )
    values = np.asarray(data.values, dtype=float).ravel()
    present = values[np.isfinite(values)]
    out["count"] = int(present.size)
    out["nulls"] = int(values.size - present.size)
    if present.size:
        out.update(
            min=_plain(present.min()),
            max=_plain(present.max()),
            mean=_plain(present.mean()),
            std=_plain(present.std(ddof=1)) if present.size > 1 else None,
            quantiles={str(q): _plain(np.quantile(present, q)) for q in QUANTILES},
        )
    else:
        out["note"] = "No values in the selection."
    return out


# --- the tools ---------------------------------------------------------------------------


def _describe_one(table: dict[str, Any], sheet: str | None) -> dict[str, Any]:
    if table["format"] == "netcdf":
        return _describe_grid(table)
    parts = table["parts"]
    out: dict[str, Any] = {
        "table": table["name"],
        "path": parts[0]["label"],
        "format": table["format"],
        "files": len(parts),
        "size_bytes": _table_size(table),
        "sheets": _sheets(table),
    }
    if len(parts) > 1:
        # Several files are one table: every row says which file it came from.
        out["source_column"] = SOURCE_COLUMN
        out["paths"] = [p["label"] for p in parts[:MAX_LIST]]
    if parts[0].get("zip"):
        out["zip"] = parts[0]["zip"]
    if parts[0].get("sqlite_table"):
        out["database"] = parts[0]["path"]
    if table["format"] in ("xlsx", "xls"):
        # 1-based, as the spreadsheet numbers its rows.
        out["header_row"] = _excel_header(str(_path(parts[0])), table["format"], sheet) + 1
    time_frame: pd.DataFrame | None = None
    union: pd.DataFrame | None = None
    if table["format"] == "parquet":
        union, rows = _parquet_scan(table["name"])
        frame = _coerce_times(_read(table, None, None, 5))
        out.update(rows=rows, sampled=False)
        # The range needs the whole time column, which is cheap: it is one column.
        stamp = _time_name_from_schema(table) or _time_column(frame)
        if stamp:
            time_frame = _coerce_times(_read(table, None, [stamp]))
    elif _too_big(table) and parts[0].get("zip"):
        # Unpacking a large member just to look at its start is the cost this refuses.
        out.update(
            rows=None,
            sampled=True,
            columns=[],
            time=None,
            note=(
                "Inside a zip and too large to open ("
                + _size_phrase(table)
                + "). Publish it as a file of its own to query it."
            ),
        )
        return out
    elif _too_big(table):
        frame = _coerce_times(_read(table, sheet, None, SAMPLE_ROWS))
        out.update(
            rows=_sqlite_shape(str(_path(parts[0])), parts[0]["sqlite_table"])[0]
            if table["format"] == "sqlite"
            else None,
            sampled=True,
            note=(
                "Too large to scan ("
                + _size_phrase(table)
                + "): columns and types come from the first "
                + str(SAMPLE_ROWS)
                + " rows, and query, aggregate and stats are not available."
            ),
        )
    else:
        frame = _whole(table["name"], sheet)
        out.update(rows=int(len(frame)), sampled=False)
    names = [str(c) for c in (union.columns if union is not None else frame.columns)]
    dtypes = {str(c): _dtype(union[c]) for c in union.columns} if union is not None else {}
    dtypes.update({str(c): _dtype(frame[c]) for c in frame.columns})
    out["columns"] = [{"name": c, "dtype": dtypes[c]} for c in names][:500]
    info, meta = _column_meta(table, names)
    for column in out["columns"]:
        column.update(info.get(column["name"], {}))
    if meta["sources"]:
        out["metadata"] = {"sources": meta["sources"]}
    if meta["description"]:
        out["description"] = meta["description"]
    if table["name"] in _PARTIAL:
        out["columns_missing_in_some_files"] = _PARTIAL[table["name"]]
    time_frame = frame if time_frame is None else time_frame
    column = _time_column(time_frame)
    out["time"] = None
    if column is not None:
        series = time_frame[column].dropna()
        out["time"] = {
            "column": column,
            "from": _plain(series.min()) if len(series) else None,
            "to": _plain(series.max()) if len(series) else None,
            "time_zone": _zone(time_frame, column),
        }
        if out["sampled"]:
            out["time"]["of"] = "the first " + str(SAMPLE_ROWS) + " rows"
    if out["time"] and out["time"]["time_zone"] is None:
        out["time"]["note"] = "The file gives no time zone for this column; none is assumed."
    return out


def describe(inp: dict[str, Any]) -> dict[str, Any]:
    tables = _tables()
    sheet = inp.get("sheet")
    if inp.get("table") is not None or len(tables) == 1:
        out: dict[str, Any] = {"tables": [_describe_one(_pick(inp), sheet)]}
    else:
        chosen = list(tables.values())[:MAX_TABLES_DESCRIBED]
        described: list[dict[str, Any]] = []
        for one in chosen:
            try:
                described.append(_describe_one(one, None))
            except (_InputError, _DataError) as exc:
                # One table that cannot be read must not hide the others.
                described.append({"table": one["name"], "error": str(exc)})
        out = {"tables": described}
        rest = list(tables)[MAX_TABLES_DESCRIBED : MAX_LIST + MAX_TABLES_DESCRIBED]
        if rest:
            out["more_tables"] = rest
    if _NOTES:
        # About a zip it opened: what it left alone and why.
        out["notes"] = [name + ": " + note for name, note in _NOTES.items()]
    return out


def _pick_rows(inp: dict[str, Any]) -> dict[str, Any]:
    """A table of rows: a gridded file has variables, not columns, and is read by slice."""
    table = _pick(inp)
    if table["format"] == "netcdf":
        raise _InputError(
            "Table "
            + table["name"]
            + " is a NetCDF file: it has variables, not columns. Use slice."
        )
    return table


def head(inp: dict[str, Any]) -> dict[str, Any]:
    table = _pick_rows(inp)
    count = _int(inp, "n", 10, 1, MAX_HEAD)
    frame = _coerce_times(_read(table, inp.get("sheet"), None, count))
    return {"columns": [str(c) for c in frame.columns], "rows": _rows(frame.head(count))}


def query(inp: dict[str, Any]) -> dict[str, Any]:
    table = _pick_rows(inp)
    columns = _names(inp, "columns", 100)
    limit = _int(inp, "limit", 100, 1, MAX_ROWS)
    frame = _prepared(table, inp, columns)
    _check_columns(frame, columns)
    keep, zone = _mask(frame, inp)
    total = int(keep.sum())
    chosen = frame.loc[keep, columns] if columns else frame.loc[keep]
    shown = chosen.head(limit)
    out: dict[str, Any] = {
        "columns": [str(c) for c in shown.columns],
        "rows": _rows(shown),
        "returned": int(len(shown)),
        "total_matching": total,
        "truncated": total > limit,
    }
    if zone or inp.get("time_zone"):
        out["time_zone"] = inp.get("time_zone") or zone
    return out


def aggregate(inp: dict[str, Any]) -> dict[str, Any]:
    table = _pick_rows(inp)
    group_by = _names(inp, "group_by", MAX_GROUP_BY)
    measures = inp.get("measures")
    if not isinstance(measures, list) or not 1 <= len(measures) <= MAX_MEASURES:
        raise _InputError("measures must be a list of 1 to " + str(MAX_MEASURES) + " items.")
    pairs: list[tuple[str, str]] = []
    for item in measures:
        if (
            not isinstance(item, dict)
            or item.get("agg") not in AGGS
            or not isinstance(item.get("column"), str)
        ):
            raise _InputError("Each measure needs column and agg (" + ", ".join(AGGS) + ").")
        pairs.append((item["column"], item["agg"]))
    step = inp.get("resample")
    if step is not None and (not isinstance(step, str) or not RESAMPLE.match(step)):
        raise _InputError("resample must look like 15min, 1h, 1D, 1MS or 1YS.")
    limit = _int(inp, "limit", 100, 1, MAX_ROWS)

    frame = _prepared(table, inp, group_by + [c for c, _ in pairs])
    _check_columns(frame, group_by + [c for c, _ in pairs])
    for column, agg in pairs:
        if agg != "count" and not pd.api.types.is_numeric_dtype(frame[column]):
            raise _InputError(
                agg + " needs a number column; " + column + " is " + _dtype(frame[column]) + "."
            )
    keep, zone = _mask(frame, inp)
    frame = frame.loc[keep]

    keys: list[Any] = list(group_by)
    time_name = None
    if step is not None:
        time_name = _time_column(frame, inp.get("time_column"))
        if time_name is None:
            raise _InputError(
                "This table has no time column to resample. Name one with time_column."
            )
        zone = _zone(frame, time_name)
        keys = [pd.Grouper(key=time_name, freq=step), *group_by]
    spec: dict[str, list[str]] = {}
    for column, agg in pairs:
        spec.setdefault(column, []).append(agg)
    try:
        if keys:
            grouped = frame.groupby(keys, dropna=False).agg(spec)
            grouped.columns = [c + "_" + a for c, a in grouped.columns]
            grouped = grouped.reset_index()
        else:
            row = {c + "_" + a: getattr(frame[c], a)() for c, a in pairs}
            grouped = pd.DataFrame([row])
    except (ValueError, TypeError) as exc:
        raise _InputError("That aggregate could not be computed: " + str(exc)[:160]) from None
    total = int(len(grouped))
    shown = grouped.head(limit)
    out: dict[str, Any] = {
        "columns": [str(c) for c in shown.columns],
        "rows": _rows(shown),
        "returned": int(len(shown)),
        "total_groups": total,
        "truncated": total > limit,
    }
    if zone or inp.get("time_zone"):
        out["time_zone"] = inp.get("time_zone") or zone
    return out


def stats(inp: dict[str, Any]) -> dict[str, Any]:
    table = _pick_rows(inp)
    column = inp.get("column")
    if not isinstance(column, str):
        raise _InputError("column must be a column name.")
    frame = _prepared(table, inp, [column])
    _check_columns(frame, [column])
    keep, zone = _mask(frame, inp)
    series = frame.loc[keep, column]
    out: dict[str, Any] = {
        "column": column,
        "dtype": _dtype(series),
        "rows": int(len(series)),
        "nulls": int(series.isna().sum()),
    }
    values = series.dropna()
    out["count"] = int(len(values))
    if len(values) == 0:
        pass
    elif pd.api.types.is_bool_dtype(series):
        out["true"] = int(values.sum())
    elif pd.api.types.is_numeric_dtype(series):
        out.update(
            min=_plain(values.min()),
            max=_plain(values.max()),
            mean=_plain(values.mean()),
            std=_plain(values.std()),
            quantiles={str(q): _plain(values.quantile(q)) for q in QUANTILES},
        )
    elif _is_time(series):
        out.update(min=_plain(values.min()), max=_plain(values.max()))
    else:
        counts = values.astype(str).value_counts().head(10)
        out.update(unique=int(values.nunique()), top=[[k, int(v)] for k, v in counts.items()])
    if zone or inp.get("time_zone"):
        out["time_zone"] = inp.get("time_zone") or zone
    return out


HANDLERS = {
    "slice": slice_values,
    "describe": describe,
    "head": head,
    "query": query,
    "aggregate": aggregate,
    "stats": stats,
}


# --- the contract ------------------------------------------------------------------------


def _error(status_code: int, code: str, message: str) -> JSONResponse:
    return JSONResponse(
        status_code=status_code,
        content={"powerai": "1", "ok": False, "error": {"code": code, "message": message}},
    )


@app.get("/health")
def health() -> dict[str, Any]:
    return {"powerai": "1", "ok": True}


@app.post("/run")
async def run(request: Request) -> JSONResponse:
    try:
        try:
            payload = await request.json()
        except Exception:  # noqa: BLE001
            raise _InputError("Request body must contain valid JSON.") from None
        if not isinstance(payload, dict):
            raise _InputError("Request body must be a JSON object.")
        if payload.get("powerai") != "1":
            raise _InputError("powerai must be the string '1'.")
        tool = payload.get("tool")
        if tool not in HANDLERS:
            raise _InputError(
                "Unknown tool " + str(tool) + ". Available: " + ", ".join(TOOLS) + "."
            )
        body = payload.get("input")
        if body is None:
            body = {}
        if not isinstance(body, dict):
            raise _InputError("input must be an object.")
        output = HANDLERS[tool](body)
        return JSONResponse(status_code=200, content={"powerai": "1", "ok": True, "output": output})
    except _InputError as exc:
        return _error(400, "invalid_input", str(exc))
    except _DataError as exc:
        return _error(500, "internal", str(exc))
    except Exception:  # noqa: BLE001
        return _error(500, "internal", "The request could not be completed.")
''')

import os, re, subprocess, sys, time, urllib.request

PORT = 8011
MODULE = "powerai_dataset_runner"
CLOUDFLARED = "/usr/local/bin/cloudflared"

# Re-running this cell has to be safe: a previous run leaves a server and a tunnel
# alive, and the next one would otherwise pass the health probe anyway because the
# *stale* server answers it — a stale process serving stale code, reported as fresh.
for pattern in ("cloudflared tunnel", f"uvicorn {MODULE}:app"):
    subprocess.run(["pkill", "-f", pattern], capture_output=True)
time.sleep(1)

if not os.path.exists(CLOUDFLARED):
    got = subprocess.run(
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/"
        f"cloudflared-linux-amd64 -O {CLOUDFLARED} && chmod +x {CLOUDFLARED}",
        shell=True, capture_output=True, text=True,
    )
    if got.returncode != 0:
        raise RuntimeError(f"could not install cloudflared:\n{got.stderr[-1000:]}")
    print("cloudflared installed")
else:
    print("cloudflared already present")

# --app-dir puts WORKDIR on the import path for this one process, so the archive
# need not be pip-installed (and may not even be a package) to be served.
# A src/ layout keeps its packages one folder down; both folders go on the path.
env = dict(os.environ, PYTHONPATH=os.pathsep.join(
    [WORKDIR, os.path.join(WORKDIR, "src"), os.environ.get("PYTHONPATH", "")]
))
# The server's own output goes to a file, not nowhere: a file that fails on import, or hides an
# error behind a plain "internal" reply, leaves its cause here.
LOG = os.path.join(os.path.dirname(WORKDIR), "endpoint.log")
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", f"{MODULE}:app", "--app-dir", WORKDIR,
     "--host", "127.0.0.1", "--port", str(PORT)],
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT, env=env,
)

healthy = False
for _ in range(90):
    if server.poll() is not None:
        break  # it exited: nothing to wait for
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=1)
        healthy = True
        break
    except Exception:
        time.sleep(1)
if not healthy:
    try:
        output = open(LOG).read()[-3000:]
    except OSError:
        output = ""
    raise RuntimeError(
        f"{MODULE}.py did not come up as a healthy server on :{PORT}. Its output:\n{output}\n"
        "This is the archive's own code, not the Hub's."
    )
print(f"endpoint up on :{PORT}")
print(f"Server log: {LOG}  (after a failed call, run  !tail -n 40 {LOG}  to see why)")

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
PUBLIC_URL = ""
deadline = time.time() + 90
while time.time() < deadline:
    line = tunnel.stdout.readline()
    if not line:
        break
    found = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if found:
        PUBLIC_URL = found.group(0)
        break
if not PUBLIC_URL:
    raise RuntimeError("no tunnel URL; re-run this cell")

ENDPOINT_URL = PUBLIC_URL + "/run"
print()
print("Endpoint address:")
print("   ", ENDPOINT_URL)

PASTE_LINE = f"Run {PRIMITIVE} at {ENDPOINT_URL}"
print()
print("Paste this line into the agent chat:")
print("   ", PASTE_LINE)
try:
    import json
    from IPython.display import HTML, display

    _line = json.dumps(PASTE_LINE).replace("<", chr(92) + "u003c")
    display(HTML(
        '<button id="powerai-copy" style="padding:6px 12px;cursor:pointer">Copy the line</button>'
        ' <span id="powerai-copied"></span>'
        "<script>(function () {"
        " var line = " + _line + ";"
        " var button = document.getElementById('powerai-copy');"
        " var note = document.getElementById('powerai-copied');"
        " function fallback() {"
        "  var box = document.createElement('textarea'); box.value = line;"
        "  document.body.appendChild(box); box.select();"
        "  try { document.execCommand('copy'); note.textContent = ' Copied.'; }"
        "  catch (e) { note.textContent = ' Select the line above and copy it.'; }"
        "  document.body.removeChild(box);"
        " }"
        " button.onclick = function () {"
        "  if (navigator.clipboard && navigator.clipboard.writeText) {"
        "   navigator.clipboard.writeText(line).then(function () { note.textContent = ' Copied.'; }, fallback);"
        "  } else { fallback(); }"
        " };"
        "})();</script>"
    ))
except Exception:
    pass  # the plain line above is enough


Copy the line printed above (use the Copy button, or select it) and paste it into the agent chat on [the Agent tab](https://hub.powerai.ai/zz-windfm-test/building-data-genome-project-2/agent).